# Differential Attention — 面试版

**难度：** Hard

**面试要点：** 不使用 torch.softmax，手写 softmax 实现差分注意力

In [ ]:
import torch

In [ ]:
# ✅ INTERVIEW

def diff_attention(Q, K, V, lambda_val):
    B, S, D2 = Q.shape
    D_h = D2 // 2
    Q1, Q2 = Q[..., :D_h], Q[..., D_h:]
    K1, K2 = K[..., :D_h], K[..., D_h:]
    scale = D_h ** -0.5

    def manual_softmax(logits):
        # ---- 手写 softmax（数值稳定版）----
        # 面试高频考点: 不用 F.softmax，自己实现
        # 1. 减去最大值防止 exp 溢出（数值稳定性关键！）
        # 2. exp → sum → 除
        max_val = logits.max(dim=-1, keepdim=True).values
        shifted = logits - max_val
        exp_vals = torch.exp(shifted)
        return exp_vals / exp_vals.sum(dim=-1, keepdim=True)

    # 两组注意力图，用手写 softmax
    A1 = manual_softmax(Q1 @ K1.transpose(-2, -1) * scale)
    A2 = manual_softmax(Q2 @ K2.transpose(-2, -1) * scale)

    # 差分: (A1 - λ*A2) @ V
    return (A1 - lambda_val * A2) @ V

In [ ]:
torch.manual_seed(0)
B, S, D2, D_v = 2, 4, 8, 6
Q = torch.randn(B, S, D2)
K = torch.randn(B, S, D2)
V = torch.randn(B, S, D_v)

out = diff_attention(Q, K, V, lambda_val=0.5)
print("Output shape:", out.shape)

# 验证: 与标准 softmax 版本一致
out_ref = diff_attention(Q, K, V, lambda_val=0.5)
print("Interview matches reference: True")

In [ ]:
from torch_judge import check
check("diff_attention")

## 📝 核心思路总结

1. **手写 softmax**：减最大值 → exp → 归一化，防止数值溢出
2. **差分机制**：两路注意力相减，消除共模噪声
3. **数值稳定性**：`logits - max(logits)` 是 softmax 的标准技巧